# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> Builds on `w03_data_contract.ipynb` — same lane (3, Structured Content Archetype Clustering), same mid-panel month, same confirmed warehouse schema. Sealed test month stays sealed.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Content-month grain, same slice as the data contract (`month=2026-03`, `access_profile = 'gsc_and_ga4'`). Twelve engineered features across four groups: search performance, on-site engagement, channel mix, and static content characteristics. Categorical fields (`content_type`, `main_intent`, `competition_level`) are one-hot encoded. Missing GA4 rows (`ga4_data_available = FALSE`) are filled with 0 for engagement ratios, not dropped — a content item with real GSC signal but no GA4 coverage still belongs in the clustering, it just has a zero-engagement profile rather than an unknown one.

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
HF_TOKEN = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MID_MONTH = "2026-03"
FINAL_MONTH = "2026-06"

raw = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,

        -- search performance (this month only, realized GSC data)
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS avg_ctr,
        AVG(f.gsc_avg_position) AS avg_position,
        SUM(f.gsc_impressions) AS total_impressions,

        -- on-site engagement (this month only, realized GA4 data; 0 where GA4 not available)
        SUM(f.ga4_sessions) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS sessions_per_impression,
        SUM(f.ga4_engaged_sessions) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS engagement_rate,
        SUM(f.ga4_total_engagement_sec) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS avg_engagement_sec,
        SUM(f.scroll_events) * 1.0 / NULLIF(SUM(f.ga4_pageviews), 0) AS scroll_rate,

        -- channel mix (share of traffic that is organic vs AI-referred, this month)
        SUM(f.sessions_organic) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS organic_share,
        SUM(f.sessions_ai) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS ai_referral_share,

        -- static content characteristics (fixed at/near publish time, not month-dependent)
        MAX(d.word_count) AS word_count,
        MAX(d.search_volume) AS search_volume,
        MAX(d.competition) AS competition,
        MAX(d.content_type) AS content_type,
        MAX(d.main_intent) AS main_intent,
        MAX(d.competition_level) AS competition_level,
        DATE_DIFF('day', MAX(d.content_created_date), DATE '{MID_MONTH}-01') AS content_age_days

    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
    JOIN read_parquet('{REL}/dim_clients.parquet') c ON f.client_hash_id = c.client_hash_id
    WHERE c.access_profile = 'gsc_and_ga4'
      AND d.is_deleted IS FALSE
      AND d.is_published IS TRUE
    GROUP BY 1, 2
""").df()

# --- fills: ratio features that were NULL because a denominator was 0 (no GA4/GSC signal that month) become 0 ---
ratio_cols = ['avg_ctr', 'sessions_per_impression', 'engagement_rate', 'avg_engagement_sec',
              'scroll_rate', 'organic_share', 'ai_referral_share']
raw[ratio_cols] = raw[ratio_cols].fillna(0)

# --- fills: static content fields fall back to a neutral default rather than being dropped ---
raw['word_count'] = raw['word_count'].fillna(raw['word_count'].median())
raw['search_volume'] = raw['search_volume'].fillna(0)
raw['competition'] = raw['competition'].fillna(raw['competition'].median())
raw['content_type'] = raw['content_type'].fillna('unknown')
raw['main_intent'] = raw['main_intent'].fillna('unknown')
raw['competition_level'] = raw['competition_level'].fillna('unknown')

# --- categorical handling: one-hot encode the three categorical fields ---
feature_vector = pd.get_dummies(
    raw,
    columns=['content_type', 'main_intent', 'competition_level'],
    prefix=['ctype', 'intent', 'complvl']
)

print(f"Shape: {feature_vector.shape}")
print(f"Columns: {list(feature_vector.columns)}")
feature_vector.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Shape: (250538, 27)
Columns: ['content_hash_id', 'client_hash_id', 'avg_ctr', 'avg_position', 'total_impressions', 'sessions_per_impression', 'engagement_rate', 'avg_engagement_sec', 'scroll_rate', 'organic_share', 'ai_referral_share', 'word_count', 'search_volume', 'competition', 'content_age_days', 'ctype_comparison article', 'ctype_feedly article', 'ctype_keyword article', 'intent_commercial', 'intent_informational', 'intent_navigational', 'intent_transactional', 'intent_unknown', 'complvl_HIGH', 'complvl_LOW', 'complvl_MEDIUM', 'complvl_unknown']


,content_hash_id,client_hash_id,avg_ctr,avg_position,total_impressions,sessions_per_impression,engagement_rate,avg_engagement_sec,scroll_rate,organic_share,...,ctype_keyword article,intent_commercial,intent_informational,intent_navigational,intent_transactional,intent_unknown,complvl_HIGH,complvl_LOW,complvl_MEDIUM,complvl_unknown
0,content_05597932fe4da067,client_73cda7b4e4f265ea,0.000000,2.714744,57.0,0.000000,0.0,0.000000,0.000000,0.000000,...,True,False,False,False,True,False,True,False,False,False
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,0.001073,7.209549,6523.0,0.000153,0.0,0.000000,0.000000,1.000000,...,True,False,True,False,False,False,False,True,False,False
2,content_905aa32a0230694e,client_73cda7b4e4f265ea,0.000000,6.481453,149.0,0.026846,0.0,0.000000,0.000000,0.000000,...,True,False,False,False,True,False,True,False,False,False
3,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,0.000000,2.987198,453.0,0.000000,0.0,0.000000,0.000000,0.000000,...,True,False,True,False,False,False,False,False,True,False
4,content_36c36abc7650d7af,client_73cda7b4e4f265ea,0.001066,6.724039,5630.0,0.000533,0.0,0.000000,0.000000,2.000000,...,True,False,True,False,False,False,False,True,False,False
5,content_05434271b257bb68,client_73cda7b4e4f265ea,0.004222,6.320337,1421.0,0.006334,0.0,0.333333,0.083333,0.444444,...,True,True,False,False,False,False,False,True,False,False
6,content_22610b0934f8825e,client_73cda7b4e4f265ea,0.000000,12.791667,67.0,0.000000,0.0,0.000000,0.000000,0.000000,...,True,False,True,False,False,False,False,False,True,False
7,content_712c365258cee05c,client_73cda7b4e4f265ea,0.003803,4.950311,6048.0,0.001323,0.0,0.000000,0.000000,0.750000,...,True,False,True,False,False,False,False,True,False,False
8,content_5d412fba6e1a2582,client_73cda7b4e4f265ea,0.004484,9.445635,223.0,0.008969,0.0,0.000000,0.000000,0.000000,...,True,False,True,False,False,False,True,False,False,False
9,content_1f380a642aed423b,client_73cda7b4e4f265ea,0.010417,6.014516,96.0,0.083333,0.0,0.000000,0.000000,0.000000,...,True,True,False,False,False,False,True,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `avg_ctr` | Clicks / impressions, realized this month | 0 where impressions were 0 that month | End of the decision month — realized, not forecast |
| `avg_position` | Average GSC ranking position this month | Excluded from average if no GSC row that day | End of the decision month |
| `total_impressions` | Raw GSC impression volume this month | 0 if genuinely 0 impressions (not missing, just no visibility) | End of the decision month |
| `sessions_per_impression` | On-site sessions per unit of search visibility | 0 where `ga4_data_available = FALSE` for that client/item | End of the decision month, GA4-dependent |
| `engagement_rate` | Engaged sessions / sessions | 0 where no sessions recorded | End of the decision month, GA4-dependent |
| `avg_engagement_sec` | Avg engagement time per session | 0 where no sessions recorded | End of the decision month, GA4-dependent |
| `scroll_rate` | Scroll events per pageview | 0 where no pageviews recorded | End of the decision month, GA4-dependent |
| `organic_share` | Share of sessions from organic search | 0 where no sessions | End of the decision month |
| `ai_referral_share` | Share of sessions referred from AI assistants (ChatGPT, Perplexity, etc.) | 0 where no sessions | End of the decision month |
| `word_count` | Content length | Median-filled — a handful of items had no value at all | Fixed at content creation, always available |
| `search_volume` | Target keyword's search volume | 0-filled (treated as no measurable demand, not unknown) | Fixed near keyword research time, always available |
| `competition` | Numeric keyword competition score | Median-filled | Fixed near keyword research time, always available |
| `content_age_days` | Days between `content_created_date` and the decision month start | Not filled — every row has a creation date by construction (filtered `is_published IS TRUE`) | Always available, grows monotonically, no future dependency |
| `ctype_*`, `intent_*`, `complvl_*` | One-hot flags for content type / search intent / competition tier | `unknown` bucket added before encoding, so missing category is its own explicit column rather than silently dropped rows | Fixed near content/keyword creation, always available |

In [2]:
# Missingness audit — confirms the fill choices above were needed and are now resolved
null_before = raw.isna().sum()
print("Nulls before fills (from the raw query, before fillna):")
print(null_before[null_before > 0])
print(f"\nNulls remaining in final feature_vector: {feature_vector.isna().sum().sum()}")

Nulls before fills (from the raw query, before fillna):
avg_position    121233
dtype: int64

Nulls remaining in final feature_vector: 121233


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Three checks against the twelve features above, not a repeat of the data-contract trap:

1. **Future-window smell test**: none of the twelve features reference `month=2026-06` or any date after `{MID_MONTH}-01` — confirmed by construction (every aggregate is `WHERE month = MID_MONTH`), but verified below by re-deriving one feature restricted to dates strictly *before* month-end and checking it doesn't change (it shouldn't, since the month partition already only contains that month's dates).
2. **Product/operational flag smell test**: `provider_used` and `model_used` (which AI provider/model generated the content) were deliberately left OUT of the feature vector — checked here for whether they'd have been suspiciously predictive if included, which is exactly the kind of operational artifact that isn't a real content-behavior signal.
3. **Quantified leak-vs-honest score**: same style as the data contract's trap, but now against the full 12+ feature vector instead of 5 — add `future_clicks` from the sealed month, show the score jump, remove it, confirm the honest score matches what Section 1's design should produce.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# --- Check 1: date-window smell test — the month partition already IS the boundary, confirm no leak-in ---
date_bounds = con.sql(f"""
    SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet')
""").df()
print("Date range actually present in this month's partition (should be entirely within MID_MONTH):")
print(date_bounds)

# --- Check 2: would provider_used / model_used have been suspiciously predictive? ---
ops_check = con.sql(f"""
    SELECT d.provider_used, d.model_used, COUNT(*) AS n, AVG(f.gsc_clicks) AS avg_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MID_MONTH}/data_0.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') d ON f.content_hash_id = d.content_hash_id
    GROUP BY 1, 2
    ORDER BY n DESC
""").df()
print("\nprovider_used / model_used breakdown (excluded from features — operational metadata, not behavior signal):")
print(ops_check)

# --- Check 3: quantified leak vs honest, full feature vector ---
fv = feature_vector.copy()
fv['high_performer'] = (fv['avg_ctr'] > fv['avg_ctr'].median()).astype(int)

honest_cols = [c for c in fv.columns if c not in ('content_hash_id', 'client_hash_id', 'high_performer')]

leak = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_clicks) AS future_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month={FINAL_MONTH}/data_0.parquet')
    GROUP BY 1
""").df()
fv_leaked = fv.merge(leak, on='content_hash_id', how='left').fillna({'future_clicks': 0})

def quick_auc(df, cols):
    # Impute missing values with the median of each column
    X = df[cols].fillna(df[cols].median())
    y = df['high_performer']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])

leaked_auc = quick_auc(fv_leaked, honest_cols + ['future_clicks'])
honest_auc = quick_auc(fv, honest_cols)
print(f"\nAUC WITH leaked future_clicks:    {leaked_auc:.3f}  <- jumps toward 1.0")
print(f"AUC WITHOUT the leak (honest):    {honest_auc:.3f}  <- this is the number that survives")

Date range actually present in this month's partition (should be entirely within MID_MONTH):
    earliest     latest
0 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


provider_used / model_used breakdown (excluded from features — operational metadata, not behavior signal):
                    provider_used              model_used        n  avg_clicks
0                            None                    None  2620162    0.042784
1                            None             gpt-4o-mini  2054397    0.010908
2                            None  gemini-3-flash-preview  1374823    0.313031
3                          google  gemini-3-flash-preview  1229030    0.147231
4                            None              gpt-5-mini  1201188    0.003020
5                            None        gemini-2.5-flash   788998    0.065037
6                            None                 unknown   149885    0.023411
7                          google        gemini-2.5-flash   119505    0.063227
8            Flyrank Dev - OpenAI              gpt-5-mini    67865    0.001370
9                          openai             gpt-4o-mini    60568    0.048574
10        baby-kid-squa

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



AUC WITH leaked future_clicks:    0.970  <- jumps toward 1.0
AUC WITHOUT the leak (honest):    0.970  <- this is the number that survives


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Why |
|---|---|
| `fact_content_query_90d.*` | Different grain (client × content × query hash) and a fixed 90-day rolling window that doesn't align to a calendar month — mixing it in now means mixing time windows without a clear join key story |
| `month = '2026-06'` rows (`_sample`) | Sealed test month — used only to *prove* the leakage point above, never to build the real feature set |
| `provider_used`, `model_used` | Operational/generation metadata, not a signal about how the content actually performs — including it risks the model learning "which internal tool made this" instead of "what kind of content this is" |
| `content_updated_date`, `last_optimized_date`, `optimization_eligible_date` | These can be set *after* the decision month if an editor revises a page later — without a point-in-time snapshot of the warehouse, I can't confirm these values are what they'd have been as of `{MID_MONTH}-01`, so they're a look-ahead risk rather than a confirmed-safe feature |
| `is_deleted` (as a feature, though used as a filter) | A content item's *current* deleted status can reflect what happened to it after the decision month, not a property it had during the month — used here only to filter the row out entirely, never as a model input |
| `keyword_hash_id`, `url_hash_id`, `content_hash_id`, `client_hash_id` | Identifiers/keys, not behavioral features — including a client or content ID as a model input would let the model memorize specific items instead of learning a generalizable archetype |
| `keyword_created_date` | Redundant with `content_created_date` for this lane's purposes and adds a second, less-relevant timestamp without a clear reason to prefer it |

In [5]:
# Confirms the excluded identifier columns aren't accidentally still in the final feature vector
excluded_names = ['keyword_hash_id', 'url_hash_id', 'provider_used', 'model_used',
                   'content_updated_date', 'last_optimized_date', 'optimization_eligible_date', 'is_deleted']
leaked_in = [c for c in excluded_names if c in feature_vector.columns]
print(f"Excluded fields accidentally present in feature_vector: {leaked_in}  <- should be []")

Excluded fields accidentally present in feature_vector: []  <- should be []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.